# M8 vehicle detectors on the full UVH-26 validation split (step 16)

Scores IISc's YOLO11-S (as released and BatchNorm-repaired) and our two fine-tunes (v2, v3) on **all 5,297 labelled UVH-26 validation frames**,
and gives IISc's model the 14-class COCO evaluation. Targets are fixed in `baselines/2026-10-27_m8_uvh26_fullval/TARGETS_LOCKED.md`.

**Before running:** upload the folder `uvh26_fullval_kaggle/bundle` as a Kaggle dataset (or put `bundle` next to this notebook in Colab), turn **Internet on**
(frames come from huggingface.co). CPU is fine: about 1 to 2 hours. A GPU is not used.

**Steps:** 1 setup, 2 smoke test on 40 frames (about 2 min; stops here if the download is broken), 3 the full detection run (resumable: re-run the cell after a disconnect),
4 score and zip. Send back `uvh26_fullval_results.zip`.

In [ ]:
import glob, os, subprocess, sys, shutil, time, zipfile, json
assert sys.version_info >= (3, 11), f"needs Python 3.11+, have {sys.version}"
hits = glob.glob("/kaggle/input/**/m8_detector_fullval.py", recursive=True) + glob.glob("**/m8_detector_fullval.py", recursive=True) \
     + glob.glob("/content/**/m8_detector_fullval.py", recursive=True)
assert hits, "bundle not found: add the uvh26_fullval_kaggle/bundle folder as a dataset (Kaggle) or upload it (Colab)"
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
WORK = f"{BASE}/uvh26_fullval"; os.makedirs(WORK, exist_ok=True)
print("bundle:", ROOT, "\nwork:", WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "opencv-python-headless", "pycocotools", "numpy"], check=True)

def run(args, cwd=ROOT):
    p = subprocess.Popen([sys.executable, *args], cwd=cwd, env=dict(os.environ, PYTHONPATH=ROOT),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    assert code == 0, f"exit {code}"

r = subprocess.run(["curl", "-sSI", "-o", "/dev/null", "-w", "%{http_code}", "https://huggingface.co/datasets/iisc-aim/UVH-26/resolve/main/UVH-26-Val/UVH-26-MV-Val.json"],
                   capture_output=True, text=True)
print("huggingface.co ->", r.stdout or r.stderr)
assert r.stdout.strip() in ("200", "302", "307"), "huggingface.co not reachable: turn Internet on for this notebook"

In [ ]:
MODELS = ["--model", "iisc=models/iisc.onnx", "--model", "iisc_bnfix=models/iisc_bnfix.onnx",
          "--model", "ft_v2=models/ft_v2.onnx", "--model", "ft_v3=models/ft_v3.onnx",
          "--model14", "iisc", "--model14", "iisc_bnfix", "--manifest", "data/dataset_manifest.json"]
DETECT = ["scripts/m8_detector_fullval.py", "detect", "--labels", "data/UVH-26-MV-Val.json", *MODELS]
SCORE = ["scripts/m8_detector_fullval.py", "score", "--labels", "data/UVH-26-MV-Val.json",
         "--old250", "data/val_sample.json", "--fresh250", "data/fresh_names.json",
         "--ref", "iisc=refs/iisc.json", "--ref", "iisc_bnfix=refs/iisc_bnfix.json",
         "--ref", "ft_v2=refs/ft_v2.json", "--ref", "ft_v3=refs/ft_v3.json"]
# 2. smoke test: 40 frames, own output folder
shutil.rmtree(f"{WORK}/smoke", ignore_errors=True)
run([*DETECT, "--limit", "40", "--shard", "20", "--tmp", f"{WORK}/tmp_smoke", "--out", f"{WORK}/smoke"])
run([*SCORE, "--run", f"{WORK}/smoke", "--boot", "200", "--out", f"{WORK}/smoke_res"])

In [ ]:
# 3. the full run. Resumable: finished 250-frame shards are skipped, so after a disconnect just run this cell again.
t0 = time.time()
run([*DETECT, "--shard", "250", "--workers", "6", "--providers", "auto", "--tmp", f"{WORK}/tmp", "--out", f"{WORK}/run"])
print(f"detection finished in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# 4. score (same script and locked settings as when I score the zip) and package
run([*SCORE, "--run", f"{WORK}/run", "--boot", "2000", "--out", f"{WORK}/res"])
ZIP = f"{BASE}/uvh26_fullval_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("run", "res"):
        for p in sorted(glob.glob(f"{WORK}/{d}/*")):
            if os.path.isfile(p):
                z.write(p, f"{d}/{os.path.basename(p)}")
print(ZIP, round(os.path.getsize(ZIP) / 1e6), "MB")